# 07 — Modelo económico de flipping inmobiliario

## 1. Objetivo

Evaluar la viabilidad económica de potenciales oportunidades de flipping inmobiliario en departamentos publicados en CABA, incorporando costos de adquisición, refacción y comercialización.

En las notebooks anteriores se construyó un Precio Esperado Reciclado (PER), utilizado para estimar el valor potencial de una propiedad luego de su refacción.

Sin embargo, la diferencia entre el PER y el precio publicado representa únicamente un potencial bruto y no contempla los costos necesarios para realizar la operación.

En esta etapa se busca:

1. Definir supuestos económicos explícitos y modificables.
2. Estimar los costos de compra, refacción, mantenimiento y venta.
3. Calcular la inversión total, la ganancia estimada y el retorno sobre la inversión (ROI).
4. Evaluar escenarios alternativos para analizar la sensibilidad de los resultados.
5. Identificar y priorizar oportunidades económicamente atractivas, considerando la calidad de los datos y la incertidumbre de las estimaciones.

**Alcance:** el modelo constituye una herramienta exploratoria de evaluación de inversiones. Sus resultados dependen de supuestos económicos y del PER estimado, por lo que no representan tasaciones profesionales ni garantías de rentabilidad.


## 2. Estructura del modelo económico

Para cada propiedad se estimarán los siguientes componentes:

### Ingresos

- **Precio de venta estimado:** se utilizará el PER base calculado anteriormente, sujeto a ajustes de escenario.

### Costos

- **Precio de adquisición:** precio publicado del inmueble, utilizado inicialmente como aproximación al precio de compra.
- **Gastos de adquisición:** comisiones, escrituración y otros gastos asociados a la compra.
- **Costo de refacción:** presupuesto estimado en función de la superficie y el alcance de la obra.
- **Costos de mantenimiento:** expensas, servicios y otros gastos durante el período de tenencia.
- **Gastos de venta:** comisiones y costos asociados a la comercialización.

### Indicadores económicos

**1. Ganancia neta estimada (USD)**

Ganancia neta = Precio de venta estimado − Costos totales

**2. Retorno sobre la inversión (ROI)**

ROI (%) = (Ganancia neta / Inversión total) × 100

La **inversión total** incluye el precio de adquisición, los gastos de compra, la refacción y los costos de mantenimiento.

Los gastos de venta se descuentan al calcular la ganancia neta, pero no se incluyen en el denominador del ROI.

### Supuestos del modelo

Los parámetros económicos se definirán como supuestos explícitos y modificables.

Se evaluarán distintos escenarios para analizar cómo cambian la ganancia neta y el ROI ante variaciones en los costos de refacción, los gastos de operación y el precio de venta.

**Importante:** los resultados representan estimaciones exploratorias y no deben interpretarse como rentabilidades garantizadas.


In [52]:

import pandas as pd
import numpy as np
from pathlib import Path

# Cargar dataset procesado
ruta_datos = Path("data/processed/remax_deptos_features.csv")

df = pd.read_csv(ruta_datos)

print("Dimensiones del dataset:", df.shape)

# Buscar columnas relacionadas con el modelo económico
palabras_clave = [
    "precio",
    "per",
    "superficie",
    "sup_",
    "expensa",
    "estado",
    "potencial",
    "barrio"
]

columnas_economicas = [
    col for col in df.columns
    if any(palabra in col.lower() for palabra in palabras_clave)
]

print("\nColumnas económicas disponibles:")
for col in columnas_economicas:
    print("-", col)

print("\nVista previa:")
display(df[columnas_economicas].head())


Dimensiones del dataset: (9499, 100)

Columnas económicas disponibles:
- Operacion
- Precio_Valor
- Moneda_Expensas
- Expensas
- Precio_m2_Total
- Precio_m2_Cubierto
- Superficie_Total_m2
- Superficie_Cubierta_m2
- Barrio
- Estado_Publicacion
- Expensas_Informadas
- Barrio_Oficial
- Estado_Propiedad
- Estado_Agrupado
- Superficie_Descubierta_m2
- Flag_Superficie_Homogeneizada_Imputada
- Superficie_Homogeneizada_m2
- Banda_Superficie
- Precio_Esperado
- PER_USD_m2_P25
- PER_USD_m2_Base
- PER_USD_m2_P75
- PER_N_Comparables
- PER_Nivel
- PER_Conservador
- PER_Base
- PER_Optimista
- Potencial_Bruto_Conservador_pct
- Potencial_Bruto_Base_pct
- Potencial_Bruto_Optimista_pct
- Expensas_por_m2
- Expensas_Imputadas
- Flag_Expensas_Imputadas

Vista previa:


,Operacion,Precio_Valor,Moneda_Expensas,Expensas,Precio_m2_Total,Precio_m2_Cubierto,Superficie_Total_m2,Superficie_Cubierta_m2,Barrio,Estado_Publicacion,...,PER_Nivel,PER_Conservador,PER_Base,PER_Optimista,Potencial_Bruto_Conservador_pct,Potencial_Bruto_Base_pct,Potencial_Bruto_Optimista_pct,Expensas_por_m2,Expensas_Imputadas,Flag_Expensas_Imputadas
0,sale,79000.0,ARS,190000.0,1832.95,1832.95,43.10,43.10,Villa Urquiza,active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4408.352668,190000.0,False
1,sale,75000.0,ARS,150000.0,1875.00,1875.00,40.00,40.00,San Nicolás,active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3750.000000,150000.0,False
2,sale,68000.0,ARS,130000.0,1511.11,1511.11,45.00,45.00,Mataderos,active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2888.888889,130000.0,False
3,sale,199000.0,ARS,252000.0,2329.39,2669.35,85.43,74.55,Palermo,active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3150.393799,252000.0,False
4,sale,69000.0,ARS,177000.0,1811.02,1874.49,38.10,36.81,San Cristobal,active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4725.670805,177000.0,False



## 3. Disponibilidad de datos para el modelo económico

Antes de estimar costos y rentabilidades, se analiza la disponibilidad de las variables necesarias para evaluar las operaciones de flipping.

Se verifica la cobertura del Precio Esperado Reciclado (PER), el precio publicado, la superficie cubierta, las expensas y la clasificación del estado del inmueble.

Este análisis permite identificar qué propiedades pueden evaluarse y cuáles presentan información faltante que requiere tratamiento adicional.

La disponibilidad de un PER no implica necesariamente que una propiedad sea elegible para inversión: posteriormente se aplicarán controles sobre su estado, uso, comparables y potencial económico.


In [53]:

# Variables necesarias para el modelo económico
variables_modelo = [
    "Precio_Valor",
    "PER_Base",
    "PER_Conservador",
    "PER_Optimista",
    "Superficie_Cubierta_m2",
    "Expensas_Imputadas",
    "Estado_Propiedad",
    "PER_N_Comparables"
]

# Disponibilidad de cada variable
disponibilidad = pd.DataFrame({
    "Variable": variables_modelo,
    "Disponibles": [
        df[col].notna().sum()
        for col in variables_modelo
    ],
    "Faltantes": [
        df[col].isna().sum()
        for col in variables_modelo
    ]
})

disponibilidad["Cobertura_pct"] = (
    disponibilidad["Disponibles"] / len(df) * 100
).round(2)

print("Disponibilidad de variables:")
display(disponibilidad)

# Condiciones mínimas para calcular rentabilidad
condiciones = (
    df["Precio_Valor"].gt(0)
    & df["PER_Base"].gt(0)
    & df["Superficie_Cubierta_m2"].gt(0)
)

print("\nPropiedades totales:", len(df))
print("Con datos mínimos para modelo económico:", condiciones.sum())
print("Sin datos mínimos:", (~condiciones).sum())

print("\nDistribución por estado de las propiedades evaluables:")
display(
    df.loc[condiciones, "Estado_Propiedad"]
    .value_counts()
    .rename_axis("Estado")
    .reset_index(name="Cantidad")
)


Disponibilidad de variables:


,Variable,Disponibles,Faltantes,Cobertura_pct
0,Precio_Valor,9499,0,100.00
1,PER_Base,428,9071,4.51
2,PER_Conservador,428,9071,4.51
3,PER_Optimista,428,9071,4.51
4,Superficie_Cubierta_m2,9498,1,99.99
5,Expensas_Imputadas,9499,0,100.00
6,Estado_Propiedad,9499,0,100.00
7,PER_N_Comparables,428,9071,4.51



Propiedades totales: 9499
Con datos mínimos para modelo económico: 428
Sin datos mínimos: 9071

Distribución por estado de las propiedades evaluables:


,Estado,Cantidad
0,Necesita_obra,428



## 4. Construcción de la base económica

Se construye una base de trabajo con las propiedades que cuentan con los datos mínimos necesarios para evaluar una operación de flipping:

- Precio publicado positivo.
- PER base disponible y positivo.
- Superficie cubierta positiva.

El universo resultante está compuesto por 428 publicaciones, todas clasificadas automáticamente como `Necesita_obra`.

Estas propiedades constituyen candidatos preliminares para el análisis económico, pero su inclusión no implica que su estado físico ni su potencial de inversión hayan sido verificados.

Se conservarán los tres escenarios de PER (conservador, base y optimista), las variables de superficie, las expensas y los indicadores de calidad del benchmark.

El dataset original permanecerá sin modificaciones.


In [54]:

# Construir base de propiedades evaluables
df_flip = df.loc[condiciones].copy()

# Verificar integridad de la base
assert len(df_flip) == 428
assert df_flip["Precio_Valor"].gt(0).all()
assert df_flip["PER_Base"].gt(0).all()
assert df_flip["Superficie_Cubierta_m2"].gt(0).all()

print("Propiedades en la base económica:", len(df_flip))

# Resumen descriptivo de variables principales
variables_resumen = [
    "Precio_Valor",
    "PER_Conservador",
    "PER_Base",
    "PER_Optimista",
    "Superficie_Cubierta_m2",
    "Expensas_Imputadas",
    "Potencial_Bruto_Base_pct",
    "PER_N_Comparables"
]

resumen = df_flip[variables_resumen].describe().T

display(
    resumen[
        ["count", "mean", "50%", "min", "max"]
    ].round(2)
)

print("\nCantidad de propiedades por barrio:")
display(
    df_flip["Barrio"]
    .value_counts()
    .head(10)
)


Propiedades en la base económica: 428


,count,mean,50%,min,max
Precio_Valor,428.0,134668.03,89000.00,27000.00,1150000.00
PER_Conservador,428.0,140967.19,98787.40,26850.36,1163423.28
PER_Base,428.0,159701.21,113065.96,30272.09,1388832.49
PER_Optimista,428.0,182755.65,127893.92,34688.81,1665103.73
Superficie_Cubierta_m2,428.0,72.54,54.93,15.00,513.00
Expensas_Imputadas,428.0,272211.67,190250.49,0.00,1800000.00
Potencial_Bruto_Base_pct,428.0,28.73,23.11,-64.74,227.46
PER_N_Comparables,428.0,18.55,16.00,10.00,45.00



Cantidad de propiedades por barrio:


,count
Barrio,
Balvanera,49
Recoleta,38
Palermo,32
Caballito,29
Belgrano,27
Almagro,24
San Nicolás,19
Retiro,17
Flores,16



## 5. Definición de supuestos económicos

El modelo económico incorpora costos que no se encuentran directamente disponibles en el dataset inmobiliario.

Por este motivo, se utilizarán parámetros explícitos y modificables, diferenciando los valores observados de los supuestos de simulación.

### 5.1. Costos de adquisición

Se consideran los gastos asociados a la compra del inmueble, expresados como porcentaje del precio de adquisición.

El precio publicado se utiliza inicialmente como aproximación al precio efectivo de compra. La posible negociación del precio podrá evaluarse mediante escenarios.

### 5.2. Costos de refacción

El costo de obra se estimará multiplicando la superficie cubierta del inmueble por un presupuesto de refacción en USD/m².

Se analizarán distintos niveles de costo para representar obras de diferente alcance y evaluar la sensibilidad de la rentabilidad.

Este presupuesto es una aproximación: no reemplaza una inspección técnica ni un presupuesto profesional.

### 5.3. Costos de mantenimiento

Se considerará un período estimado de tenencia durante el cual el inversor deberá afrontar expensas y otros gastos.

Las expensas se encuentran expresadas principalmente en pesos argentinos, por lo que será necesario establecer un tipo de cambio de referencia para convertirlas a dólares.

### 5.4. Costos de comercialización

Se incorporarán los gastos asociados a la venta, calculados como porcentaje del precio de venta estimado.

### 5.5. Escenarios

Se evaluarán tres escenarios:

- **Conservador:** menor precio de venta esperado y mayores costos de refacción.
- **Base:** PER base y costos de refacción intermedios.
- **Optimista:** mayor precio de venta esperado y menores costos de refacción.

Los escenarios permitirán analizar la sensibilidad de la ganancia neta y del ROI, sin asumir que los resultados futuros son conocidos con certeza.



## 6. Parámetros iniciales del modelo económico

Se establece un conjunto de supuestos económicos preliminares para simular operaciones de flipping.

Los valores son ilustrativos y se utilizan para explorar la sensibilidad de los resultados. No representan cotizaciones verificadas, tarifas obligatorias ni presupuestos profesionales.

| Parámetro | Supuesto inicial |
|---|---:|
| Gastos de adquisición | 4% del precio de compra |
| Refacción económica | 150 USD/m² |
| Refacción intermedia | 300 USD/m² |
| Refacción costosa | 500 USD/m² |
| Período de tenencia | 6 meses |
| Gastos de venta | 3% del precio de venta |
| Otros gastos de mantenimiento | 100 USD/mes |
| Descuento sobre precio publicado | 0% |

Se utilizarán los tres escenarios de PER existentes en el dataset.

El costo de refacción se calculará sobre la superficie cubierta y se combinará con los escenarios de venta:

- **Conservador:** PER conservador y refacción costosa.
- **Base:** PER base y refacción intermedia.
- **Optimista:** PER optimista y refacción económica.

Los gastos de adquisición, venta y mantenimiento permanecerán constantes entre escenarios en esta primera aproximación.

Las expensas se incorporarán posteriormente, una vez definido y documentado el tipo de cambio ARS/USD.

Los supuestos deberán contrastarse con fuentes del mercado antes de interpretar los resultados como estimaciones de rentabilidad realistas.


In [55]:

# Supuestos económicos preliminares
# Valores ilustrativos, pendientes de validación externa

supuestos = {
    "gastos_compra_pct": 0.04,
    "gastos_venta_pct": 0.03,
    "meses_tenencia": 6,
    "otros_gastos_mensuales_usd": 100,
    "descuento_compra_pct": 0.00,
    "refaccion_economica_usd_m2": 150,
    "refaccion_intermedia_usd_m2": 300,
    "refaccion_costosa_usd_m2": 500
}

tabla_supuestos = pd.DataFrame(
    [
        {"Parametro": clave, "Valor": valor}
        for clave, valor in supuestos.items()
    ]
)

print("Parámetros iniciales del modelo:")
display(tabla_supuestos)

# Verificaciones
assert 0 <= supuestos["gastos_compra_pct"] < 1
assert 0 <= supuestos["gastos_venta_pct"] < 1
assert supuestos["meses_tenencia"] > 0
assert supuestos["refaccion_economica_usd_m2"] > 0

assert (
    supuestos["refaccion_economica_usd_m2"]
    < supuestos["refaccion_intermedia_usd_m2"]
    < supuestos["refaccion_costosa_usd_m2"]
)

print("Supuestos definidos correctamente.")


Parámetros iniciales del modelo:


,Parametro,Valor
0,gastos_compra_pct,0.04
1,gastos_venta_pct,0.03
2,meses_tenencia,6.00
3,otros_gastos_mensuales_usd,100.00
4,descuento_compra_pct,0.00
5,refaccion_economica_usd_m2,150.00
6,refaccion_intermedia_usd_m2,300.00
7,refaccion_costosa_usd_m2,500.00


Supuestos definidos correctamente.



## 7. Tratamiento de expensas y conversión monetaria

El modelo económico utiliza dólares estadounidenses (USD) como moneda de referencia, dado que los precios publicados y las estimaciones de PER están expresados en esta moneda.

Sin embargo, las expensas pueden estar informadas en pesos argentinos (ARS) o en otras monedas.

Antes de incorporarlas al cálculo de rentabilidad, se verificará la moneda original, la distribución de los valores y la existencia de registros imputados.

Para las expensas expresadas en ARS se utilizará un tipo de cambio de referencia explícito, con fecha y fuente documentadas.

La conversión se realizará mediante:

**Expensas mensuales (USD) = Expensas mensuales (ARS) / Tipo de cambio (ARS por USD)**

Se mantendrá un indicador para distinguir los valores informados de los imputados.

El tipo de cambio será un supuesto fijo de la simulación y no una proyección de su evolución durante el período de tenencia.


In [56]:

# Revisar moneda y calidad de las expensas
print("Monedas de expensas en las 428 propiedades:")
display(
    df_flip["Moneda_Expensas"]
    .fillna("Sin informar")
    .value_counts(dropna=False)
)

print("\nExpensas imputadas:")
display(
    df_flip["Flag_Expensas_Imputadas"]
    .value_counts(dropna=False)
)

print("\nDistribución de expensas por moneda:")
resumen_expensas = (
    df_flip
    .assign(
        Moneda_Expensas=df_flip["Moneda_Expensas"]
        .fillna("Sin informar")
    )
    .groupby("Moneda_Expensas")["Expensas_Imputadas"]
    .agg(["count", "median", "min", "max"])
    .round(2)
)

display(resumen_expensas)

print("\nExpensas iguales a cero:",
      (df_flip["Expensas_Imputadas"] == 0).sum())

print("Expensas negativas:",
      (df_flip["Expensas_Imputadas"] < 0).sum())


Monedas de expensas en las 428 propiedades:


,count
Moneda_Expensas,
ARS,428



Expensas imputadas:


,count
Flag_Expensas_Imputadas,
False,401
True,27



Distribución de expensas por moneda:


,count,median,min,max
Moneda_Expensas,,,,
ARS,428,190250.49,0.0,1800000.0



Expensas iguales a cero: 1
Expensas negativas: 0



## 8. Conversión de expensas a USD

Las 428 propiedades evaluables presentan expensas expresadas en pesos argentinos (ARS).

De estas, 401 tienen expensas informadas y 27 presentan valores imputados. Se identifica además una propiedad con expensas iguales a cero, que deberá considerarse como posible anomalía.

Para incorporar las expensas al modelo económico se utilizará un tipo de cambio ARS/USD de referencia, indicando su fuente y fecha.

**Expensas mensuales USD = Expensas mensuales ARS / Tipo de cambio ARS/USD**

**Costo de expensas durante la tenencia = Expensas mensuales USD × Meses de tenencia**

La conversión supone un tipo de cambio constante durante el período analizado. No contempla inflación futura, variaciones cambiarias ni incrementos de expensas.

Se conservará el indicador de imputación para evaluar posteriormente la confiabilidad de los resultados.


In [57]:

# Tipo de cambio MEP de referencia
tipo_cambio_ars_usd = 1575.00
fecha_tipo_cambio = "2026-11-02"
fuente_tipo_cambio = "DolarHub - histórico dólar MEP, venta"
url_fuente = "https://dolarhub.com/cotizaciones/dolar-mep"

# Conversión de expensas
df_flip["Expensas_USD_Mensuales"] = (
    df_flip["Expensas_Imputadas"] / tipo_cambio_ars_usd
)

# Costo de expensas durante la tenencia
df_flip["Costo_Expensas_Tenencia_USD"] = (
    df_flip["Expensas_USD_Mensuales"]
    * supuestos["meses_tenencia"]
)

print("Tipo de cambio:", tipo_cambio_ars_usd, "ARS/USD")
print("Fecha:", fecha_tipo_cambio)
print("Fuente:", fuente_tipo_cambio)

display(
    df_flip[
        ["Expensas_USD_Mensuales",
         "Costo_Expensas_Tenencia_USD"]
    ].describe().round(2)
)


Tipo de cambio: 1575.0 ARS/USD
Fecha: 2026-11-02
Fuente: DolarHub - histórico dólar MEP, venta


,Expensas_USD_Mensuales,Costo_Expensas_Tenencia_USD
count,428.00,428.00
mean,172.83,1037.00
std,156.37,938.25
min,0.00,0.00
25%,82.54,495.24
50%,120.79,724.76
75%,195.92,1175.50
max,1142.86,6857.14



## 9. Cálculo de rentabilidad neta

Se estima la rentabilidad de cada una de las 428 propiedades candidatas mediante tres escenarios económicos.

### Componentes del modelo

**Precio de compra efectivo = Precio publicado × (1 − Descuento de compra)**

**Gastos de adquisición = Precio de compra efectivo × Porcentaje de gastos de compra**

**Costo de refacción = Superficie cubierta × Costo de refacción por m²**

**Costo de mantenimiento = Expensas durante la tenencia + Otros gastos mensuales × Meses de tenencia**

**Inversión total = Precio de compra efectivo + Gastos de adquisición + Costo de refacción + Costo de mantenimiento**

**Gastos de venta = Precio de venta estimado × Porcentaje de gastos de venta**

**Ganancia neta = Precio de venta estimado − Gastos de venta − Inversión total**

**ROI (%) = Ganancia neta / Inversión total × 100**

### Escenarios

- **Conservador:** PER conservador y refacción de 500 USD/m².
- **Base:** PER base y refacción de 300 USD/m².
- **Optimista:** PER optimista y refacción de 150 USD/m².

Los resultados representan simulaciones condicionadas a los supuestos definidos, no ganancias garantizadas.

El modelo no incorpora por separado impuestos sobre eventuales ganancias, costos financieros, contingencias extraordinarias ni diferencias entre precios publicados y precios efectivos de cierre. Estas limitaciones deberán considerarse al interpretar la rentabilidad.


In [58]:

# Calcular componentes comunes a los tres escenarios

df_flip["Precio_Compra_Efectivo"] = (
    df_flip["Precio_Valor"]
    * (1 - supuestos["descuento_compra_pct"])
)

df_flip["Gastos_Compra_USD"] = (
    df_flip["Precio_Compra_Efectivo"]
    * supuestos["gastos_compra_pct"]
)

df_flip["Otros_Gastos_Tenencia_USD"] = (
    supuestos["otros_gastos_mensuales_usd"]
    * supuestos["meses_tenencia"]
)

df_flip["Costo_Mantenimiento_USD"] = (
    df_flip["Costo_Expensas_Tenencia_USD"]
    + df_flip["Otros_Gastos_Tenencia_USD"]
)

# Configuración de escenarios
escenarios = {
    "Conservador": {
        "columna_per": "PER_Conservador",
        "costo_obra_m2": supuestos["refaccion_costosa_usd_m2"]
    },
    "Base": {
        "columna_per": "PER_Base",
        "costo_obra_m2": supuestos["refaccion_intermedia_usd_m2"]
    },
    "Optimista": {
        "columna_per": "PER_Optimista",
        "costo_obra_m2": supuestos["refaccion_economica_usd_m2"]
    }
}

for nombre, config in escenarios.items():

    df_flip[f"Refaccion_{nombre}_USD"] = (
        df_flip["Superficie_Cubierta_m2"]
        * config["costo_obra_m2"]
    )

    df_flip[f"Inversion_Total_{nombre}_USD"] = (
        df_flip["Precio_Compra_Efectivo"]
        + df_flip["Gastos_Compra_USD"]
        + df_flip[f"Refaccion_{nombre}_USD"]
        + df_flip["Costo_Mantenimiento_USD"]
    )

    df_flip[f"Gastos_Venta_{nombre}_USD"] = (
        df_flip[config["columna_per"]]
        * supuestos["gastos_venta_pct"]
    )

    df_flip[f"Ganancia_Neta_{nombre}_USD"] = (
        df_flip[config["columna_per"]]
        - df_flip[f"Gastos_Venta_{nombre}_USD"]
        - df_flip[f"Inversion_Total_{nombre}_USD"]
    )

    df_flip[f"ROI_{nombre}_pct"] = (
        df_flip[f"Ganancia_Neta_{nombre}_USD"]
        / df_flip[f"Inversion_Total_{nombre}_USD"]
        * 100
    )

# Resumen de resultados por escenario
resumen_escenarios = []

for nombre in escenarios:
    roi = df_flip[f"ROI_{nombre}_pct"]
    ganancia = df_flip[f"Ganancia_Neta_{nombre}_USD"]

    resumen_escenarios.append({
        "Escenario": nombre,
        "ROI_mediano_pct": roi.median(),
        "ROI_promedio_pct": roi.mean(),
        "Ganancia_mediana_USD": ganancia.median(),
        "Propiedades_ROI_positivo": (roi > 0).sum(),
        "Porcentaje_ROI_positivo": (roi > 0).mean() * 100
    })

tabla_resultados = pd.DataFrame(resumen_escenarios)

print("Resultados económicos de los 428 candidatos:")
display(tabla_resultados.round(2))


Resultados económicos de los 428 candidatos:


,Escenario,ROI_mediano_pct,ROI_promedio_pct,Ganancia_mediana_USD,Propiedades_ROI_positivo,Porcentaje_ROI_positivo
0,Conservador,-21.46,-20.37,-23502.56,46,10.75
1,Base,-2.94,-0.64,-3189.62,190,44.39
2,Optimista,18.20,22.62,17285.46,340,79.44



## 10. Evaluación de robustez económica

Se analiza la sensibilidad de la rentabilidad frente a diferentes escenarios de precio de reventa y costo de refacción.

Las propiedades se clasifican en cuatro grupos:

- **Rentable en los tres escenarios:** presenta ROI positivo incluso bajo el escenario conservador.
- **Rentable en base y optimista:** presenta ROI positivo en los escenarios base y optimista, pero no en el conservador.
- **Rentable solo en optimista:** necesita condiciones favorables para obtener ganancias.
- **No rentable:** presenta ROI no positivo en los tres escenarios.

Esta clasificación permite distinguir oportunidades económicamente más resistentes a cambios en los supuestos.

La robustez económica no garantiza la viabilidad de la inversión: las publicaciones todavía requieren validación de su estado físico, destino de uso, calidad del PER y costos reales de refacción.


In [59]:

# Clasificación de robustez económica

roi_c = df_flip["ROI_Conservador_pct"]
roi_b = df_flip["ROI_Base_pct"]
roi_o = df_flip["ROI_Optimista_pct"]

condiciones_robustez = [
    (roi_c > 0) & (roi_b > 0) & (roi_o > 0),
    (roi_c <= 0) & (roi_b > 0) & (roi_o > 0),
    (roi_b <= 0) & (roi_o > 0),
    (roi_o <= 0)
]

categorias_robustez = [
    "Rentable en los tres escenarios",
    "Rentable en base y optimista",
    "Rentable solo en optimista",
    "No rentable"
]

df_flip["Robustez_Economica"] = np.select(
    condiciones_robustez,
    categorias_robustez,
    default="Revisar"
)

tabla_robustez = (
    df_flip["Robustez_Economica"]
    .value_counts()
    .reindex(categorias_robustez, fill_value=0)
    .rename_axis("Clasificacion")
    .reset_index(name="Cantidad")
)

tabla_robustez["Porcentaje"] = (
    tabla_robustez["Cantidad"] / len(df_flip) * 100
).round(2)

print("Clasificación de robustez económica:")
display(tabla_robustez)

print("\nROI base por clasificación:")
display(
    df_flip.groupby("Robustez_Economica")["ROI_Base_pct"]
    .agg(["count", "median", "mean", "min", "max"])
    .round(2)
)


Clasificación de robustez económica:


,Clasificacion,Cantidad,Porcentaje
0,Rentable en los tres escenarios,46,10.75
1,Rentable en base y optimista,144,33.64
2,Rentable solo en optimista,150,35.05
3,No rentable,88,20.56



ROI base por clasificación:


,count,median,mean,min,max
Robustez_Economica,,,,,
No rentable,88,-21.80,-25.18,-69.26,-10.80
Rentable en base y optimista,144,9.69,10.81,0.12,52.41
Rentable en los tres escenarios,46,33.10,37.63,10.47,110.64
Rentable solo en optimista,150,-8.66,-8.96,-27.18,-0.06



## 11. Calidad del benchmark y priorización preliminar

La rentabilidad estimada no es suficiente para identificar una oportunidad de inversión confiable.

Se analizará la calidad del Precio Esperado Reciclado (PER), considerando la cantidad de propiedades comparables y el nivel de agrupación utilizado para construir el benchmark.

Una mayor cantidad de comparables puede aportar estabilidad a la estimación, aunque no garantiza por sí sola que los inmuebles sean equivalentes.

También se revisará la relación entre robustez económica y calidad del PER.

Antes de construir un ranking, se identificarán los valores disponibles de `PER_Nivel` y su distribución, evitando asignar ponderaciones arbitrarias sin comprender su significado.

Las propiedades seleccionadas seguirán sujetas a revisión manual del anuncio, destino de uso y estado real del inmueble.


In [60]:

# Inspección de calidad del PER

print("Distribución de niveles del PER:")
display(
    df_flip["PER_Nivel"]
    .fillna("Sin nivel")
    .value_counts(dropna=False)
    .rename_axis("Nivel_PER")
    .reset_index(name="Cantidad")
)

print("\nCantidad de comparables por nivel:")
display(
    df_flip.groupby("PER_Nivel", dropna=False)["PER_N_Comparables"]
    .agg(["count", "median", "min", "max"])
    .round(2)
)

print("\nRobustez económica por nivel del PER:")
display(
    pd.crosstab(
        df_flip["PER_Nivel"],
        df_flip["Robustez_Economica"]
    )
)

print("\nComparables de las 46 propiedades robustas:")
display(
    df_flip.loc[
        df_flip["Robustez_Economica"]
        == "Rentable en los tres escenarios",
        "PER_N_Comparables"
    ].describe().round(2)
)


Distribución de niveles del PER:


,Nivel_PER,Cantidad
0,3.0,145
1,2.0,122
2,1.0,106
3,4.0,55



Cantidad de comparables por nivel:


,count,median,min,max
PER_Nivel,,,,
1.0,106,14.0,10.0,24.0
2.0,122,15.0,10.0,35.0
3.0,145,19.0,10.0,45.0
4.0,55,19.0,10.0,42.0



Robustez económica por nivel del PER:


Robustez_Economica,No rentable,Rentable en base y optimista,Rentable en los tres escenarios,Rentable solo en optimista
PER_Nivel,,,,
1.0,24,26,10,46
2.0,24,42,14,42
3.0,27,59,14,45
4.0,13,17,8,17



Comparables de las 46 propiedades robustas:


,PER_N_Comparables
count,46.00
mean,18.09
std,6.43
min,10.00
25%,14.00
50%,18.00
75%,21.00
max,41.00



## 12. Interpretación de los niveles del PER

Los 428 candidatos cuentan con un benchmark de precio estimado a partir de un mínimo de 10 comparables.

La variable `PER_Nivel` presenta cuatro categorías numéricas, cuya interpretación debe verificarse antes de incorporarla a un sistema de priorización.

No se asumirá automáticamente que un mayor número de comparables implica una estimación de mejor calidad, dado que también importa la similitud entre las propiedades utilizadas como referencia.

El objetivo de esta etapa es comprender la estructura de los niveles del PER y determinar si pueden utilizarse para evaluar la especificidad del benchmark.

Posteriormente se construirá un ranking económico preliminar que combine rentabilidad, robustez y criterios de revisión manual.


In [61]:

# Inspeccionar la relación entre niveles del PER
# y características de las propiedades

columnas_inspeccion = [
    "Barrio",
    "Barrio_Oficial",
    "Banda_Superficie",
    "Superficie_Cubierta_m2",
    "PER_Nivel",
    "PER_N_Comparables",
    "PER_USD_m2_Base"
]

print("Ejemplos de propiedades por nivel del PER:")

for nivel in sorted(df_flip["PER_Nivel"].dropna().unique()):
    print(f"\n--- PER Nivel {int(nivel)} ---")

    muestra = (
        df_flip.loc[
            df_flip["PER_Nivel"] == nivel,
            columnas_inspeccion
        ]
        .head(5)
    )

    display(muestra)

print("\nDistribución de bandas de superficie por nivel:")
display(
    pd.crosstab(
        df_flip["PER_Nivel"],
        df_flip["Banda_Superficie"]
    )
)


Ejemplos de propiedades por nivel del PER:

--- PER Nivel 1 ---


,Barrio,Barrio_Oficial,Banda_Superficie,Superficie_Cubierta_m2,PER_Nivel,PER_N_Comparables,PER_USD_m2_Base
63,Balvanera,Balvanera,Chica,53.00,1.0,15.0,1536.538462
300,Balvanera,Balvanera,Chica,34.20,1.0,11.0,1695.960530
469,Balvanera,Balvanera,Chica,34.78,1.0,20.0,1588.450292
539,Caballito,Caballito,Media,47.00,1.0,15.0,1916.793727
558,Balvanera,Balvanera,Grande,75.00,1.0,18.0,1473.626264



--- PER Nivel 2 ---


,Barrio,Barrio_Oficial,Banda_Superficie,Superficie_Cubierta_m2,PER_Nivel,PER_N_Comparables,PER_USD_m2_Base
18,Palermo Chico,Palermo,Chica,22.00,2.0,16.0,2786.292600
97,Barrio Norte,Recoleta,Chica,38.55,2.0,22.0,2593.783266
162,Recoleta,Recoleta,Chica,50.76,2.0,14.0,2615.784039
222,Retiro,San Nicolas,Media,31.00,2.0,10.0,2048.909957
266,Belgrano,Belgrano,Grande,155.00,2.0,14.0,2672.757246



--- PER Nivel 3 ---


,Barrio,Barrio_Oficial,Banda_Superficie,Superficie_Cubierta_m2,PER_Nivel,PER_N_Comparables,PER_USD_m2_Base
212,San Telmo,San Telmo,Chica,28.00,3.0,41.0,1769.230769
257,Villa Pueyrredón,Villa Pueyrredon,Media,58.00,3.0,30.0,2320.910397
325,Velez Sarsfield,Velez Sarsfield,Media,40.00,3.0,13.0,1934.624697
436,Villa Crespo,Villa Crespo,Media,37.08,3.0,16.0,2113.414038
437,Colegiales,Colegiales,Chica,50.93,3.0,24.0,2655.024237



--- PER Nivel 4 ---


,Barrio,Barrio_Oficial,Banda_Superficie,Superficie_Cubierta_m2,PER_Nivel,PER_N_Comparables,PER_USD_m2_Base
278,Palermo,Palermo,Grande,305.00,4.0,19.0,2638.830013
363,Caballito,Caballito,Grande,200.00,4.0,14.0,2072.601087
1092,Parque Patricios,Parque Patricios,Grande,80.15,4.0,39.0,1490.566038
1115,Recoleta,Recoleta,Grande,201.78,4.0,19.0,2284.263959
1160,Almagro,Almagro,Media,105.93,4.0,23.0,2013.157895



Distribución de bandas de superficie por nivel:


Banda_Superficie,Chica,Grande,Media
PER_Nivel,,,
1.0,60,27,19
2.0,55,29,38
3.0,68,37,40
4.0,12,27,16



## 13. Ranking preliminar de oportunidades de flipping

Se construye un ranking de las propiedades que presentan rentabilidad positiva en los tres escenarios económicos.

Estas propiedades se consideran candidatas económicamente robustas dentro de los supuestos de simulación utilizados.

El ranking se ordena por ROI conservador, priorizando las operaciones que mantienen mayor rentabilidad incluso bajo condiciones desfavorables.

Para cada propiedad se incluyen:

- Precio publicado e inversión total estimada.
- Ganancia neta y ROI en los escenarios conservador y base.
- Nivel de especificidad del PER.
- Cantidad de comparables utilizados.
- Barrio, superficie y enlace de publicación.

Los niveles del PER se interpretan de la siguiente manera:

- Nivel 1: barrio, ambientes, superficie y antigüedad.
- Nivel 2: barrio, ambientes y superficie.
- Nivel 3: comuna, ambientes y superficie.
- Nivel 4: comuna y ambientes.

El ranking es preliminar. Una rentabilidad elevada no garantiza que la publicación corresponda realmente a una propiedad que necesita refacción ni que el precio esperado pueda alcanzarse en una venta efectiva.

La selección final requerirá verificar manualmente el destino del inmueble, su estado físico, la vigencia del anuncio y los costos reales de la operación.


In [66]:

# Seleccionar propiedades rentables en los tres escenarios

df_ranking = df_flip.loc[
    df_flip["Robustez_Economica"]
    == "Rentable en los tres escenarios"
].copy()

# Ordenar por rentabilidad conservadora
df_ranking = df_ranking.sort_values(
    "ROI_Conservador_pct",
    ascending=False
).reset_index(drop=True)

df_ranking.insert(
    0, "Ranking", range(1, len(df_ranking) + 1)
)

# Columnas del ranking
columnas_ranking = [
    "Ranking",
    "ID",
    "Barrio",
    "Superficie_Cubierta_m2",
    "Precio_Valor",
    "Inversion_Total_Base_USD",
    "Ganancia_Neta_Conservador_USD",
    "ROI_Conservador_pct",
    "Ganancia_Neta_Base_USD",
    "ROI_Base_pct",
    "PER_Nivel",
    "PER_N_Comparables",
    "Link"
]

print("Propiedades robustas:", len(df_ranking))

print("\nTOP 15 oportunidades económicas preliminares:")
display(
    df_ranking[columnas_ranking]
    .head(15)
    .round(2)
)

print("\nDistribución por nivel del PER:")
display(
    df_ranking["PER_Nivel"]
    .value_counts()
    .sort_index()
)


Propiedades robustas: 46

TOP 15 oportunidades económicas preliminares:


,Ranking,ID,Barrio,Superficie_Cubierta_m2,Precio_Valor,Inversion_Total_Base_USD,Ganancia_Neta_Conservador_USD,ROI_Conservador_pct,Ganancia_Neta_Base_USD,ROI_Base_pct,PER_Nivel,PER_N_Comparables,Link
0,1,507319,Recoleta,184.42,260000.0,330897.43,155778.15,42.36,281000.18,84.92,1.0,14.0,https://www.remax.com.ar/listings/venta-depto-...
1,2,636825,San Cristobal,135.13,90000.0,135718.05,46612.74,28.64,150154.19,110.64,3.0,18.0,https://www.remax.com.ar/listings/comodo-depar...
2,3,644844,San Nicolás,386.00,180000.0,310457.14,108852.89,28.08,246392.86,79.36,4.0,21.0,https://www.remax.com.ar/listings/piso-de-674m...
3,4,612831,Palermo Hollywood,152.78,310000.0,370662.57,101275.87,25.24,173067.09,46.69,1.0,11.0,https://www.remax.com.ar/listings/duplex-piso-...
4,5,644478,Barrio Norte,130.01,145000.0,192364.90,50742.38,23.24,106361.23,55.29,1.0,12.0,https://www.remax.com.ar/listings/venta-consul...
5,6,629512,Recoleta,22.00,30000.0,38857.14,9506.30,21.98,18760.86,48.28,1.0,14.0,https://www.remax.com.ar/listings/venta-monoam...
6,7,569285,Centro / Microcentro,248.26,129000.0,213047.52,47202.63,17.97,134516.48,63.14,4.0,21.0,https://www.remax.com.ar/listings/venta-depto-...
7,8,509302,Monserrat,172.30,95000.0,152042.38,28579.15,15.32,89177.62,58.65,4.0,21.0,https://www.remax.com.ar/listings/departamento...
8,9,584152,Belgrano,112.00,260000.0,305704.76,42608.87,12.99,100979.48,33.03,2.0,19.0,https://www.remax.com.ar/listings/3-ambientes-...
9,10,550445,Palermo,36.00,99900.0,115696.00,15620.79,12.71,41199.53,35.61,2.0,19.0,https://www.remax.com.ar/listings/venta-depart...



Distribución por nivel del PER:


,count
PER_Nivel,
1.0,10
2.0,14
3.0,14
4.0,8



## 14. Controles de elegibilidad y alertas de revisión

El ranking económico preliminar contiene propiedades que presentan rentabilidad positiva en los tres escenarios.

Sin embargo, una estimación de rentabilidad elevada no garantiza que el inmueble sea adecuado para una estrategia de flipping residencial.

Se incorporan controles exploratorios para detectar publicaciones que requieren revisión manual:

- **Destino no residencial:** menciones a oficinas, consultorios, estudios profesionales o usos comerciales.
- **Superficie elevada:** inmuebles con más de 200 m² cubiertos, cuyo presupuesto de obra puede requerir un tratamiento específico.
- **Rentabilidad extraordinaria:** propiedades con ROI base superior al 80%, que podrían reflejar oportunidades excepcionales o problemas de estimación.
- **Benchmark menos específico:** propiedades con PER nivel 4, calculado mediante comparables agrupados por comuna y ambientes.

Estas alertas no representan errores confirmados ni implican exclusión automática.

Su finalidad es mejorar la trazabilidad de las recomendaciones y orientar la validación manual de las oportunidades priorizadas.


In [67]:

# Alertas de revisión sobre las 46 propiedades robustas

import re

# Combinar título y descripción
texto_publicacion = (
    df_ranking["Titulo"].fillna("").astype(str)
    + " "
    + df_ranking["Descripcion"].fillna("").astype(str)
).str.lower()

# Patrones de posibles usos no residenciales
patron_no_residencial = (
    r"\b(?:oficina|oficinas|consultorio|consultorios|"
    r"apto profesional|uso profesional|"
    r"uso comercial|local comercial)\b"
)

df_ranking["Alerta_Uso_No_Residencial"] = (
    texto_publicacion.str.contains(
        patron_no_residencial,
        regex=True,
        na=False
    )
)

df_ranking["Alerta_Superficie_Alta"] = (
    df_ranking["Superficie_Cubierta_m2"] > 200
)

df_ranking["Alerta_ROI_Extremo"] = (
    df_ranking["ROI_Base_pct"] > 80
)

df_ranking["Alerta_PER_Nivel_4"] = (
    df_ranking["PER_Nivel"] == 4
)

columnas_alertas = [
    "Alerta_Uso_No_Residencial",
    "Alerta_Superficie_Alta",
    "Alerta_ROI_Extremo",
    "Alerta_PER_Nivel_4"
]

df_ranking["Cantidad_Alertas"] = (
    df_ranking[columnas_alertas].sum(axis=1)
)

df_ranking["Requiere_Revision"] = (
    df_ranking["Cantidad_Alertas"] > 0
)

print("Resumen de alertas:")
display(
    df_ranking[columnas_alertas]
    .sum()
    .rename("Propiedades")
    .to_frame()
)

print("\nPropiedades con al menos una alerta:",
      df_ranking["Requiere_Revision"].sum())

print("Propiedades sin estas alertas:",
      (~df_ranking["Requiere_Revision"]).sum())

print("\nTOP 15 con alertas:")
display(
    df_ranking[
        [
            "Ranking",
            "ID",
            "Barrio",
            "ROI_Conservador_pct",
            "ROI_Base_pct",
            "PER_Nivel",
            "Cantidad_Alertas",
            "Requiere_Revision"
        ]
    ].head(15).round(2)
)


Resumen de alertas:


,Propiedades
Alerta_Uso_No_Residencial,17
Alerta_Superficie_Alta,3
Alerta_ROI_Extremo,2
Alerta_PER_Nivel_4,8



Propiedades con al menos una alerta: 22
Propiedades sin estas alertas: 24

TOP 15 con alertas:


,Ranking,ID,Barrio,ROI_Conservador_pct,ROI_Base_pct,PER_Nivel,Cantidad_Alertas,Requiere_Revision
0,1,507319,Recoleta,42.36,84.92,1.0,2,True
1,2,636825,San Cristobal,28.64,110.64,3.0,1,True
2,3,644844,San Nicolás,28.08,79.36,4.0,3,True
3,4,612831,Palermo Hollywood,25.24,46.69,1.0,0,False
4,5,644478,Barrio Norte,23.24,55.29,1.0,1,True
5,6,629512,Recoleta,21.98,48.28,1.0,1,True
6,7,569285,Centro / Microcentro,17.97,63.14,4.0,3,True
7,8,509302,Monserrat,15.32,58.65,4.0,2,True
8,9,584152,Belgrano,12.99,33.03,2.0,1,True
9,10,550445,Palermo,12.71,35.61,2.0,1,True



## 15. Priorización de oportunidades

A partir del ranking económico preliminar, se incorpora una clasificación de prioridad según las alertas detectadas.

De las 46 propiedades rentables en los tres escenarios:

- 24 no presentan ninguna de las alertas automáticas definidas.
- 22 presentan al menos una alerta relacionada con el destino de uso, superficie, rentabilidad extraordinaria o especificidad del benchmark.

Las propiedades sin alertas se priorizan para el análisis comercial, ordenadas por su ROI conservador.

Las propiedades con alertas se mantienen en una lista de revisión adicional, sin descartarlas automáticamente.

La ausencia de alertas no constituye una validación definitiva. Todas las oportunidades deberán revisarse antes de tomar decisiones de inversión.

Este procedimiento evita que una rentabilidad elevada determine por sí sola la recomendación y permite documentar las limitaciones de la selección.


In [68]:

# Crear clasificación de prioridad
df_ranking["Prioridad_Analisis"] = np.where(
    df_ranking["Requiere_Revision"],
    "Revisión adicional",
    "Prioridad de análisis"
)

# Ordenar: primero sin alertas y luego por ROI conservador
df_priorizado = df_ranking.sort_values(
    ["Requiere_Revision", "ROI_Conservador_pct"],
    ascending=[True, False]
).copy().reset_index(drop=True)

df_priorizado["Ranking_Priorizado"] = (
    np.arange(1, len(df_priorizado) + 1)
)

columnas_finales = [
    "Ranking_Priorizado",
    "ID",
    "Barrio",
    "Precio_Valor",
    "Superficie_Cubierta_m2",
    "Inversion_Total_Base_USD",
    "Ganancia_Neta_Base_USD",
    "ROI_Conservador_pct",
    "ROI_Base_pct",
    "PER_Nivel",
    "PER_N_Comparables",
    "Cantidad_Alertas",
    "Prioridad_Analisis",
    "Link"
]

print("Resumen de prioridades:")
display(
    df_priorizado["Prioridad_Analisis"]
    .value_counts()
    .rename_axis("Prioridad")
    .reset_index(name="Cantidad")
)

print("\nTOP 10 oportunidades priorizadas:")
display(
    df_priorizado[columnas_finales]
    .head(10)
    .round(2)
)

print("\nROI conservador de las oportunidades sin alertas:")
display(
    df_priorizado.loc[
        ~df_priorizado["Requiere_Revision"],
        "ROI_Conservador_pct"
    ].describe().round(2)
)


Resumen de prioridades:


,Prioridad,Cantidad
0,Prioridad de análisis,24
1,Revisión adicional,22



TOP 10 oportunidades priorizadas:


,Ranking_Priorizado,ID,Barrio,Precio_Valor,Superficie_Cubierta_m2,Inversion_Total_Base_USD,Ganancia_Neta_Base_USD,ROI_Conservador_pct,ROI_Base_pct,PER_Nivel,PER_N_Comparables,Cantidad_Alertas,Prioridad_Analisis,Link
0,1,612831,Palermo Hollywood,310000.0,152.78,370662.57,173067.09,25.24,46.69,1.0,11.0,0,Prioridad de análisis,https://www.remax.com.ar/listings/duplex-piso-...
1,2,641943,Boedo,95000.0,87.00,126338.10,40374.09,12.24,31.96,3.0,14.0,0,Prioridad de análisis,https://www.remax.com.ar/listings/venta-tres-a...
2,3,511218,Belgrano,249000.0,141.63,303763.29,77280.36,11.72,25.44,2.0,14.0,0,Prioridad de análisis,https://www.remax.com.ar/listings/depto-152m-4...
3,4,580233,Constitución,39500.0,52.81,58003.00,28274.04,11.28,48.75,3.0,24.0,0,Prioridad de análisis,https://www.remax.com.ar/listings/3-ambientes-...
4,5,518133,Villa Santa Rita,54900.0,35.72,68945.33,20020.70,8.58,29.04,3.0,18.0,0,Prioridad de análisis,https://www.remax.com.ar/listings/departamento...
5,6,634847,Boedo,52000.0,43.29,68333.67,34505.26,8.16,50.50,3.0,18.0,0,Prioridad de análisis,https://www.remax.com.ar/listings/venta-depart...
6,7,655658,Colegiales,80000.0,50.93,99955.19,39024.84,7.61,39.04,3.0,24.0,0,Prioridad de análisis,https://www.remax.com.ar/listings/venta-colegi...
7,8,604042,Palermo,45000.0,23.80,54863.81,17210.17,6.46,31.37,1.0,10.0,0,Prioridad de análisis,https://www.remax.com.ar/listings/venta-monoam...
8,9,505452,Villa Crespo,60000.0,44.00,76580.95,16985.43,5.37,22.18,2.0,12.0,0,Prioridad de análisis,https://www.remax.com.ar/listings/oportunidad-...
9,10,567781,Caballito,89000.0,69.05,114522.62,39682.70,5.26,34.65,2.0,20.0,0,Prioridad de análisis,https://www.remax.com.ar/listings/venta-depto-...



ROI conservador de las oportunidades sin alertas:


,ROI_Conservador_pct
count,24.00
mean,5.15
std,5.79
min,0.20
25%,1.17
50%,2.99
75%,7.75
max,25.24



## 16. Exportación de resultados

Se exportan los resultados del modelo económico para facilitar su reutilización en las siguientes etapas del proyecto.

Se generan dos archivos:

**Base económica completa:** contiene las 428 propiedades evaluables, sus costos estimados, rentabilidades y clasificación de robustez.

**Ranking priorizado:** contiene las 46 propiedades rentables en los tres escenarios, ordenadas según las alertas automáticas y el ROI conservador.

Los archivos permiten conectar el análisis económico con futuras visualizaciones y herramientas de apoyo a decisiones.

Los resultados son estimaciones preliminares sujetas a validación de supuestos, revisión manual de publicaciones y verificación de costos reales.


In [69]:

# Exportar resultados del modelo económico

from pathlib import Path

carpeta_salida = Path("data/processed")
carpeta_salida.mkdir(parents=True, exist_ok=True)

ruta_base = carpeta_salida / "flipping_modelo_economico.csv"
ruta_ranking = carpeta_salida / "flipping_ranking_priorizado.csv"

# Exportar base económica completa
df_flip.to_csv(
    ruta_base,
    index=False,
    encoding="utf-8-sig"
)

# Exportar ranking priorizado
df_priorizado.to_csv(
    ruta_ranking,
    index=False,
    encoding="utf-8-sig"
)

assert ruta_base.exists()
assert ruta_ranking.exists()

print("Archivos exportados correctamente:")
print(f"{ruta_base}: {len(df_flip)} propiedades")
print(f"{ruta_ranking}: {len(df_priorizado)} propiedades")


Archivos generados correctamente:
1. data/processed/flipping_modelo_economico.csv — 428 propiedades
2. data/processed/flipping_ranking_priorizado.csv — 46 propiedades

Descargando archivos...



## 17. Conclusiones y limitaciones del modelo económico

### 17.1. Principales resultados

El modelo económico de flipping permitió evaluar 428 departamentos de CABA clasificados como propiedades que necesitan refacción y que cuentan con una estimación del Precio Esperado Reciclado (PER).

Se analizaron tres escenarios de inversión, considerando distintos costos de refacción y valores esperados de reventa.

Los resultados muestran que:

- **Escenario conservador:** 46 propiedades (10,75%) presentan rentabilidad positiva, con un ROI mediano de -21,46% para el conjunto evaluado.
- **Escenario base:** 190 propiedades (44,39%) presentan rentabilidad positiva, con un ROI mediano de -2,94%.
- **Escenario optimista:** 340 propiedades (79,44%) presentan rentabilidad positiva, con un ROI mediano de 18,20%.

Se identificaron **46 propiedades rentables en los tres escenarios**, consideradas económicamente robustas bajo los supuestos utilizados.

### 17.2. Priorización de oportunidades

Sobre las 46 propiedades robustas se aplicaron controles automáticos de revisión relacionados con:

- Posibles usos no residenciales.
- Superficies cubiertas superiores a 200 m².
- Rentabilidades base superiores al 80%.
- Estimaciones PER correspondientes al nivel jerárquico 4.

Como resultado:

- **24 propiedades (52,17%)** no presentan las alertas definidas y se consideran prioritarias para el análisis comercial.
- **22 propiedades (47,83%)** presentan al menos una alerta y requieren revisión adicional.

Las 24 propiedades priorizadas representan el **5,61% de las 428 candidatas iniciales**.

Entre las propiedades sin alertas, el ROI conservador presenta una mediana de 2,99%, lo que indica que varias operaciones mantienen márgenes reducidos incluso cuando resultan rentables en los tres escenarios.

### 17.3. Interpretación del PER

El Precio Esperado Reciclado se estima mediante propiedades de referencia en buen estado o recicladas, utilizando un benchmark jerárquico con un mínimo de 10 comparables.

Los niveles considerados son:

1. Barrio, ambientes, superficie y antigüedad.
2. Barrio, ambientes y superficie.
3. Comuna, ambientes y superficie.
4. Comuna y ambientes.

Los niveles más agregados permiten ampliar la cobertura del modelo, aunque pueden incorporar propiedades menos similares.

El PER representa un valor de referencia basado en precios publicados, no un precio de venta efectivamente concretado.

### 17.4. Limitaciones

Los resultados deben interpretarse considerando las siguientes limitaciones:

1. **Precios de publicación:** tanto los precios de adquisición como las referencias de reventa provienen de anuncios inmobiliarios y pueden diferir de los precios finales de cierre.
2. **Costos estimados:** los gastos de compra, venta, refacción y mantenimiento se calculan mediante supuestos preliminares que requieren validación con fuentes de mercado.
3. **Estado de conservación:** la clasificación automática de propiedades que necesitan obra puede contener falsos positivos, según lo observado en la validación manual previa.
4. **Costos no modelados:** no se incorporan de manera exhaustiva contingencias de obra, impuestos específicos, financiación ni otros gastos extraordinarios.
5. **Tiempo de comercialización:** se supone un período de tenencia de seis meses, aunque la duración real de una operación puede ser diferente.
6. **Incertidumbre del PER:** los escenarios P25, mediana y P75 representan variabilidad entre comparables, pero no constituyen intervalos de confianza ni garantías sobre el precio futuro de venta.
7. **Alertas automáticas:** la ausencia de alertas no confirma la elegibilidad de una propiedad y las alertas detectadas no representan errores necesariamente comprobados.

### 17.5. Conclusión general

El análisis demuestra que una diferencia favorable entre el precio publicado y el precio esperado de mercado no es suficiente para identificar una oportunidad de flipping.

Al incorporar costos de adquisición, refacción, mantenimiento y venta, una parte importante del potencial bruto desaparece.

De las 428 propiedades evaluadas, únicamente 46 mantienen rentabilidad positiva en los tres escenarios y 24 no presentan las alertas automáticas definidas.

Por lo tanto, el principal aporte del modelo es transformar una identificación inicial de propiedades potencialmente subvaluadas en un **sistema de priorización económica y revisión de riesgos**.

Los resultados permiten orientar la búsqueda y la validación manual de oportunidades, pero no reemplazan una tasación profesional, un presupuesto de obra ni un análisis financiero específico de cada operación.


In [70]:

# Verificación final de consistencia

import pandas as pd

base_exportada = pd.read_csv(
    "data/processed/flipping_modelo_economico.csv"
)

ranking_exportado = pd.read_csv(
    "data/processed/flipping_ranking_priorizado.csv"
)

assert len(base_exportada) == 428
assert len(ranking_exportado) == 46

assert ranking_exportado["ID"].is_unique

assert (
    ranking_exportado["Robustez_Economica"]
    == "Rentable en los tres escenarios"
).all()

assert (ranking_exportado["ROI_Conservador_pct"] > 0).all()
assert (ranking_exportado["ROI_Base_pct"] > 0).all()
assert (ranking_exportado["ROI_Optimista_pct"] > 0).all()

assert (
    ranking_exportado["Prioridad_Analisis"]
    == "Prioridad de análisis"
).sum() == 24

assert (
    ranking_exportado["Prioridad_Analisis"]
    == "Revisión adicional"
).sum() == 22

print("VALIDACIÓN FINAL EXITOSA")
print("Base económica:", len(base_exportada), "propiedades")
print("Ranking robusto:", len(ranking_exportado), "propiedades")
print("Prioridad de análisis: 24")
print("Revisión adicional: 22")
print("Sin IDs duplicados y con ROI positivo en los tres escenarios.")


VALIDACIÓN FINAL EXITOSA
Base económica: 428 propiedades
Ranking robusto: 46 propiedades
Prioridad de análisis: 24
Revisión adicional: 22
Sin IDs duplicados y con ROI positivo en los tres escenarios.
